# FASE 2A: Modelado Clásico

**Proyecto Integrador — Grupo 2:** Análisis de Satisfacción en Productos de Oficina
**Modelos:** Regresión Logística, Random Forest, LightGBM, XGBoost

In [ ]:
# Instalacion de dependencias
!pip install mlflow -q

## Configuración del Entorno

In [ ]:
import os
import sys
import gc
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import mlflow

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

import lightgbm as lgb
import xgboost as xgb

from scipy.sparse import hstack, csr_matrix
from tqdm.auto import tqdm

RANDOM_SEED = 42

FEATURE_COLS = ['mayusculas_count', 'char_total', 'exclamacion_count',
                'interrogacion_count', 'porcentaje_mayusculas',
                'puntuacion_emocional', 'total_tokens', 'unique_types', 'ttr']

In [ ]:
IN_COLAB = 'google.colab' in sys.modules

TOTAL_MEMORY = 0

try:
    import psutil
    TOTAL_MEMORY = psutil.virtual_memory().total
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/ML/proyecto_integrador')
else:
    BASE = Path('..')

DATA_DIR = BASE / 'data'
REPORTS_DIR = BASE / 'reports'
MODELS_DIR = BASE / 'models'
DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI', 'https://humorous-trusting-domelike.ngrok-free.dev')
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment('f2a_modelado_clasico')
print(f"MLFLOW_TRACKING_URI: {MLFLOW_TRACKING_URI}")

print(f"=== Environment Info ===")
print(f"IN_COLAB: {IN_COLAB}")
if TOTAL_MEMORY:
    print(f"System RAM: {TOTAL_MEMORY / (1024**3):.1f} GB")
print(f"BASE: {BASE}")
print(f"DATA_DIR: {DATA_DIR}")

## Carga de Datos

Dataset canónico generado por `f1_eda_definitivo.ipynb`: 2.5M reseñas balanceadas.
La columna `target_class` contiene los labels: Negativo (1-2 estrellas), Neutro (3), Positivo (4-5).
Se mapean a valores numéricos 0/1/2 con LabelEncoder para sklearn.

In [ ]:
print('Cargando dataset canonico desde f1_eda_definitivo...')
CANONICAL_PATH = DATA_DIR / 'office_products_balanced.parquet'

try:
    df = pd.read_parquet(CANONICAL_PATH, columns=['text', 'rating', 'target_class'] + FEATURE_COLS)
except FileNotFoundError:
    print(f'[ERROR] Dataset canonico no encontrado en {CANONICAL_PATH}')
    print('Ejecute primero f1_eda_definitivo.ipynb en Colab para generarlo.')
    raise
except Exception as e:
    print(f'[ERROR] No se pudo cargar el dataset: {e}')
    raise

label_encoder = LabelEncoder()
df['target'] = label_encoder.fit_transform(df['target_class'])
print('Mapeo de clases:', dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# word_count filter: purgar reseñas telegraficas sin peso semantico
df['word_count'] = df['text'].astype(str).str.split().str.len()
df = df[df['word_count'] >= 5]
print(f'Registros tras filtro word_count >= 5: {len(df):,}')

print('\nDistribucion de target (0=Negativo, 1=Neutro, 2=Positivo):')
print(df['target'].value_counts().sort_index())

SUBSAMPLE_SIZE = 1_000_000
print(f'\nSubmuestreo estratificado a {SUBSAMPLE_SIZE:,} filas...')
df, _ = train_test_split(
    df, train_size=SUBSAMPLE_SIZE, random_state=RANDOM_SEED, stratify=df['target']
)
print(f'Dataset reducido a {len(df):,} filas')
print(df['target'].value_counts().sort_index())

## División Train/Test

Se divide ANTES de vectorizar para prevenir Data Leakage: el vocabulario del conjunto de prueba no debe contaminar el entrenamiento.

In [ ]:
print('Dividiendo datos en Train y Test (80/20)...')
X_train_text, X_test_text, X_train_feats, X_test_feats, y_train, y_test = train_test_split(
    df['text'], df[FEATURE_COLS], df['target'],
    test_size=0.20, random_state=RANDOM_SEED, stratify=df['target']
)

del df
gc.collect()

## Vectorización TF-IDF + Features Engineered

TF-IDF con max 10K features + 9 features lingüísticas del EDA concatenadas.

In [ ]:
print('Entrenando TF-IDF Vectorizer...')
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words='english',
    min_df=5
)

X_train_tfidf = vectorizer.fit_transform(X_train_text.astype(str))
X_test_tfidf = vectorizer.transform(X_test_text.astype(str))

print('Concatenando TF-IDF con features engineered...')
X_train = hstack([X_train_tfidf, csr_matrix(X_train_feats.values)])
X_test = hstack([X_test_tfidf, csr_matrix(X_test_feats.values)])

print(f'Matriz combinada: {X_train.shape}')
del X_train_tfidf, X_test_tfidf
gc.collect()

joblib.dump(vectorizer, MODELS_DIR / 'tfidf_vectorizer.joblib')
print(f'Vectorizer guardado en {MODELS_DIR / "tfidf_vectorizer.joblib"}')

## Modelos Base

### Regresión Logística

In [ ]:
print('Entrenando Regresion Logistica (Baseline)...')
log_model = LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1)
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)

joblib.dump(log_model, MODELS_DIR / 'logreg_model.joblib')

with mlflow.start_run(run_name='logreg_baseline', nested=True):
    mlflow.set_tag('model_type', 'logistic_regression')
    mlflow.log_param('C', 0.5)
    mlflow.log_param('max_iter', 1000)
    mlflow.log_param('feature_count', X_train.shape[1])
    mlflow.log_metric('f1_macro', f1_score(y_test, y_pred_log, average='macro'))

print(f'Modelo LogReg guardado en {MODELS_DIR / "logreg_model.joblib"}')

### Random Forest

Limitado con `max_samples=0.05` y `max_depth=30` para controlar tiempo en CPU.

In [ ]:
print('Entrenando Random Forest...')
rf_model = RandomForestClassifier(
    n_estimators=100, max_samples=0.05, max_depth=30,
    class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

joblib.dump(rf_model, MODELS_DIR / 'rf_model.joblib')

with mlflow.start_run(run_name='random_forest', nested=True):
    mlflow.set_tag('model_type', 'random_forest')
    mlflow.log_param('n_estimators', 100)
    mlflow.log_param('max_samples', 0.05)
    mlflow.log_param('max_depth', 30)
    mlflow.log_metric('f1_macro', f1_score(y_test, y_pred_rf, average='macro'))

print(f'Modelo RF guardado en {MODELS_DIR / "rf_model.joblib"}')

### LightGBM

Tuneado con `num_leaves=64`, `learning_rate=0.05`, `min_child_samples=20`.

In [ ]:
print('Entrenando LightGBM...')
lgb_model = lgb.LGBMClassifier(
    class_weight='balanced', random_state=RANDOM_SEED,
    n_jobs=-1, verbose=-1,
    num_leaves=64, learning_rate=0.05, min_child_samples=20
)
lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_test)

joblib.dump(lgb_model, MODELS_DIR / 'lgb_model.joblib')

with mlflow.start_run(run_name='lightgbm', nested=True):
    mlflow.set_tag('model_type', 'lightgbm')
    mlflow.log_param('num_leaves', 64)
    mlflow.log_param('learning_rate', 0.05)
    mlflow.log_metric('f1_macro', f1_score(y_test, y_pred_lgb, average='macro'))

print(f'Modelo LightGBM guardado en {MODELS_DIR / "lgb_model.joblib"}')

### XGBoost (CPU)

Optimizado con `tree_method='hist'` en CPU.

In [ ]:
gc.collect()

print('Entrenando XGBoost...')

classes = np.unique(y_train)
cw = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights_dict = {int(c): float(w) for c, w in zip(classes, cw)}

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    objective='multi:softprob',
    num_class=3,
    random_state=RANDOM_SEED,
    eval_metric='mlogloss',
    tree_method='hist',
)

sample_weights = np.array([class_weights_dict[y] for y in y_train])
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_xgb = xgb_model.predict(X_test)

joblib.dump(xgb_model, MODELS_DIR / 'xgb_model.joblib')

with mlflow.start_run(run_name='xgboost', nested=True):
    mlflow.set_tag('model_type', 'xgboost')
    mlflow.log_param('n_estimators', 300)
    mlflow.log_param('max_depth', 5)
    mlflow.log_param('learning_rate', 0.05)
    mlflow.log_param('tree_method', 'hist')
    mlflow.log_metric('f1_macro', f1_score(y_test, y_pred_xgb, average='macro'))

print('Entrenamiento completado.')
print(f'Modelo XGBoost guardado en {MODELS_DIR / "xgb_model.joblib"}')

## AutoML Benchmark

LazyPredict ejecuta 27 clasificadores sobre una muestra estratificada de 20K train / 5K test.

In [ ]:
from lazypredict.Supervised import LazyClassifier

print('Muestreando 20K train / 5K test para AutoML benchmark...')
X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train, y_train, train_size=20000, random_state=RANDOM_SEED, stratify=y_train
)
X_test_sample, _, y_test_sample, _ = train_test_split(
    X_test, y_test, test_size=5000, random_state=RANDOM_SEED, stratify=y_test
)

print('Iniciando Benchmark AutoML (LazyPredict)...')
print('Esto puede tomar varios minutos: evalua 27 clasificadores sobre la muestra.')
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
models_df, predictions = clf.fit(X_train_sample, X_test_sample, y_train_sample, y_test_sample)

print('\nTop 10 modelos segun LazyPredict:')
print(models_df.head(10).to_string())

lazy_path = REPORTS_DIR / 'lazypredict_results.csv'
models_df.to_csv(lazy_path)
print(f'Resultados completos guardados en {lazy_path}')

## Evaluación Comparativa

Métrica rectora: F1-macro sobre la partición de test (20% = 500K muestras). Se exportan métricas a JSON y MLflow.

In [ ]:
with mlflow.start_run(run_name='evaluation_comparativa'):
    mlflow.set_tag('phase', '2a')
    mlflow.set_tag('dataset_size', '1M_subsample')

    print('\n' + '='*60)
    print('EVALUACION FINAL - F1-macro como metrica rectora')
    print('='*60)

    models = {
        'Logistic Regression (Baseline)': y_pred_log,
        'Random Forest': y_pred_rf,
        'LightGBM': y_pred_lgb,
        'XGBoost': y_pred_xgb
    }

    results = []
    class_labels = ['Negativo', 'Neutro', 'Positivo']

    for name, preds in models.items():
        f1_macro = f1_score(y_test, preds, average='macro')
        acc = accuracy_score(y_test, preds)
        precision_macro, recall_macro, _, _ = precision_recall_fscore_support(y_test, preds, average='macro')
        cm = confusion_matrix(y_test, preds).tolist()

        p_class, r_class, f1_class, _ = precision_recall_fscore_support(y_test, preds, labels=[0, 1, 2])

        print(f"\n{'='*50}")
        print(f' {name}')
        print(f"{'='*50}")
        print(f'  F1-macro:  {f1_macro:.4f}')
        print(f'  Accuracy:  {acc:.4f}')
        print(f'  Precision macro: {precision_macro:.4f}')
        print(f'  Recall macro:    {recall_macro:.4f}')
        print('\n  Classification Report:')
        print(classification_report(y_test, preds, target_names=class_labels, digits=4))

        results.append({
            'model_name': name,
            'model_type': name.lower().replace(' ', '_').replace('(', '').replace(')', ''),
            'f1_macro': round(f1_macro, 4),
            'precision_macro': round(precision_macro, 4),
            'recall_macro': round(recall_macro, 4),
            'accuracy': round(acc, 4),
            'confusion_matrix': cm,
            'per_class': {
                cl: {
                    'precision': round(p_class[i], 4),
                    'recall': round(r_class[i], 4),
                    'f1': round(f1_class[i], 4)
                } for i, cl in enumerate(class_labels)
            }
        })

    report_path = REPORTS_DIR / 'metrics_fase2.json'
    try:
        with open(report_path, 'w') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f'\n[METRICAS] Exportadas a {report_path}')
    except Exception as e:
        print(f'[ERROR] No se pudo exportar metricas: {e}')

    print('\n' + '='*60)
    print('ANALISIS DE ERROR - Por clase')
    print('='*60)
    for name, preds in models.items():
        print(f'\n--- {name} ---')
        cm = confusion_matrix(y_test, preds)
        for i, cl in enumerate(class_labels):
            total_real = cm[i].sum()
            correctos = cm[i][i]
            print(f'  {cl:12s}: {correctos:5d}/{total_real} correctos ({correctos/total_real*100:.1f}%)')
            errores = [(j, cm[i][j]) for j in range(3) if j != i]
            for j, cnt in sorted(errores, key=lambda x: -x[1]):
                print(f'           -> {cnt:5d} clasificados como {class_labels[j]}')